In [ ]:
import pandas as pd
import re

pd.set_option('display.max_rows', 1000)

Load all the data

In [ ]:
#Links for reference

#Google Sheets 2018-2023
#2023
'https://docs.google.com/spreadsheets/d/1zlh79aLdicsDVjZoxfwVm8SY1iFlKm-GBI34TMk1I30/edit?usp=sharing'
#2022
'https://docs.google.com/spreadsheets/d/1mmsehBjl-V1eVueiCJEjFV6PhM6eKaCCPFIfcrk7IKk/edit?gid=0#gid=0'
#2021
'https://docs.google.com/spreadsheets/d/1E-Suwl9z_e8-W1JuP5HVmIKv6C4mpd73pFpgtCdrTkQ/edit?gid=0#gid=0'
#2020
'https://docs.google.com/spreadsheets/d/1Q5M7Zw_A-Kn2V7csyxGqleVXWliLVQ4e080aNI7oIiE/edit?gid=969721861#gid=969721861'
#2019
'https://docs.google.com/spreadsheets/d/1mY0ckZ7AKwlgeZv5uMA6eah7QwabyPv47SGNuqrO20I/edit?gid=969721861#gid=969721861'
#2018
'https://docs.google.com/spreadsheets/d/1nV6SU6eGIzi0_tz6ccezkEmtJewfioq3/edit?gid=1198915962#gid=1198915962'

#API endpoint 2016-2017
#2017
'https://data.colorado.gov/resource/uhi6-hddy.csv'
#2016
'https://data.colorado.gov/resource/m8vm-brgw.csv'

#Google Sheets 2007-2015
#2015
'https://docs.google.com/spreadsheets/d/1g4MnqPpjTFaYmhjIjeZUuOUdA3mkmvVntfcpD5gtRHY/edit?gid=1203697909#gid=1203697909'
#2014
'https://docs.google.com/spreadsheets/d/1Z6eI4edrGjrb2sJ_4gxYPljB7g60RkBk5BsMxyA_eH4/edit?gid=126874728#gid=126874728'
#2013
    #Intake
'https://docs.google.com/spreadsheets/d/1DjZ9cYKT9sBC1oNBD8HUuS3Zpu1m1qAVETigNPYKNqQ/edit?gid=1450511850#gid=1450511850
    #Outflow
'https://docs.google.com/spreadsheets/d/1DjZ9cYKT9sBC1oNBD8HUuS3Zpu1m1qAVETigNPYKNqQ/edit?gid=1492755857#gid=1492755857'
#2012
    #Intake
'https://docs.google.com/spreadsheets/d/1cZZFmS-o9QAwi1RrMnoy2gYiF_myL61sr6ynVfNOu6Y/edit?gid=101086594#gid=101086594'
    #Outflow
'https://docs.google.com/spreadsheets/d/1cZZFmS-o9QAwi1RrMnoy2gYiF_myL61sr6ynVfNOu6Y/edit?gid=805617810#gid=805617810'
#2011
    #Intake
'https://docs.google.com/spreadsheets/d/14Y6F_BRnpv4BtMYT4U5QPmCk8rKUB5YF1OCTmpoTkyw/edit?gid=1112172445#gid=1112172445'
    #Outflow
'https://docs.google.com/spreadsheets/d/14Y6F_BRnpv4BtMYT4U5QPmCk8rKUB5YF1OCTmpoTkyw/edit?gid=309603042#gid=309603042'
#2010
    #Intake
'https://docs.google.com/spreadsheets/d/1v0x7Pe_zYEEM0g-MyWXbpC1ZpkqpzCOTuJt3tw-y_XY/edit?gid=1457683636#gid=1457683636'
    #Outflow
'https://docs.google.com/spreadsheets/d/1v0x7Pe_zYEEM0g-MyWXbpC1ZpkqpzCOTuJt3tw-y_XY/edit?gid=1507785229#gid=1507785229'
#2009
    #Intake
'https://docs.google.com/spreadsheets/d/11A67bHzjsJ2SSIyFIoK5MNbIdlSXtORDDtcHA_adcsg/edit?gid=831397415#gid=831397415'
    #Outflow
'https://docs.google.com/spreadsheets/d/11A67bHzjsJ2SSIyFIoK5MNbIdlSXtORDDtcHA_adcsg/edit?gid=765780685#gid=765780685'
#2008
    #Intake
'https://docs.google.com/spreadsheets/d/1Av7-Wr-Lf5G3lbjz6BOJfONgdPqfD_VyeG9eBWazU6A/edit?gid=938588532#gid=938588532'
    #Outflow
'https://docs.google.com/spreadsheets/d/1Av7-Wr-Lf5G3lbjz6BOJfONgdPqfD_VyeG9eBWazU6A/edit?gid=1557235659#gid=1557235659'
#2007
'https://docs.google.com/spreadsheets/d/1CvEIJbREKwJRMFqkTOFWd2u6Bb_JZzhwLkU1JHdiYFE/edit?gid=994387581#gid=994387581'

#Licensing
'https://docs.google.com/spreadsheets/d/1q5PSJaDc-cpfKXBn6tQbg0jVxvwyrCNZUBk-5CCortA/edit?gid=531274251#gid=531274251'

In [78]:
# Helper function to convert Google Sheets URL to CSV export URL, skip converting API endpoints
def gsheet_to_csv_url(url):
    if url.endswith('.csv'):
        return url
    if "/edit" in url:
        base = url.split("/edit")[0]
        gid = "0"
        if "gid=" in url:
            gid = url.split("gid=")[-1].split("#")[0]
        return f"{base}/export?format=csv&gid={gid}"
    return url

# List of (year, url) tuples
sheet_links = [
    ("2023", "https://docs.google.com/spreadsheets/d/1zlh79aLdicsDVjZoxfwVm8SY1iFlKm-GBI34TMk1I30/edit?usp=sharing"),
    ("2022", "https://docs.google.com/spreadsheets/d/1mmsehBjl-V1eVueiCJEjFV6PhM6eKaCCPFIfcrk7IKk/edit?gid=0#gid=0"),
    ("2021", "https://docs.google.com/spreadsheets/d/1E-Suwl9z_e8-W1JuP5HVmIKv6C4mpd73pFpgtCdrTkQ/edit?gid=0#gid=0"),
    ("2020", "https://docs.google.com/spreadsheets/d/1Q5M7Zw_A-Kn2V7csyxGqleVXWliLVQ4e080aNI7oIiE/edit?gid=969721861#gid=969721861"),
    ("2019", "https://docs.google.com/spreadsheets/d/1mY0ckZ7AKwlgeZv5uMA6eah7QwabyPv47SGNuqrO20I/edit?gid=969721861#gid=969721861"),
    ("2018", "https://docs.google.com/spreadsheets/d/1nV6SU6eGIzi0_tz6ccezkEmtJewfioq3/edit?gid=1198915962#gid=1198915962"),
    ("2017", "https://data.colorado.gov/resource/uhi6-hddy.csv"), #api endpoint https://dev.socrata.com/foundry/data.colorado.gov/m8vm-brgw
    ("2016", "https://data.colorado.gov/resource/m8vm-brgw.csv"), #api endpoint https://dev.socrata.com/foundry/data.colorado.gov/uhi6-hddy
    ("2015", "https://docs.google.com/spreadsheets/d/1g4MnqPpjTFaYmhjIjeZUuOUdA3mkmvVntfcpD5gtRHY/edit?gid=1203697909#gid=1203697909"),
    ("2014", "https://docs.google.com/spreadsheets/d/1Z6eI4edrGjrb2sJ_4gxYPljB7g60RkBk5BsMxyA_eH4/edit?gid=126874728#gid=126874728"),
    ("2013 Intake", "https://docs.google.com/spreadsheets/d/1DjZ9cYKT9sBC1oNBD8HUuS3Zpu1m1qAVETigNPYKNqQ/edit?gid=1450511850#gid=1450511850"),
    ("2013 Outflow", "https://docs.google.com/spreadsheets/d/1DjZ9cYKT9sBC1oNBD8HUuS3Zpu1m1qAVETigNPYKNqQ/edit?gid=1492755857#gid=1492755857"),
    ("2012 Intake", "https://docs.google.com/spreadsheets/d/1cZZFmS-o9QAwi1RrMnoy2gYiF_myL61sr6ynVfNOu6Y/edit?gid=101086594#gid=101086594"),
    ("2012 Outflow", "https://docs.google.com/spreadsheets/d/1cZZFmS-o9QAwi1RrMnoy2gYiF_myL61sr6ynVfNOu6Y/edit?gid=805617810#gid=805617810"),
    ("2011 Intake", "https://docs.google.com/spreadsheets/d/14Y6F_BRnpv4BtMYT4U5QPmCk8rKUB5YF1OCTmpoTkyw/edit?gid=1112172445#gid=1112172445"),
    ("2011 Outflow", "https://docs.google.com/spreadsheets/d/14Y6F_BRnpv4BtMYT4U5QPmCk8rKUB5YF1OCTmpoTkyw/edit?gid=309603042#gid=309603042"),
    ("2010 Intake", "https://docs.google.com/spreadsheets/d/1v0x7Pe_zYEEM0g-MyWXbpC1ZpkqpzCOTuJt3tw-y_XY/edit?gid=1457683636#gid=1457683636"),
    ("2010 Outflow", "https://docs.google.com/spreadsheets/d/1v0x7Pe_zYEEM0g-MyWXbpC1ZpkqpzCOTuJt3tw-y_XY/edit?gid=1507785229#gid=1507785229"),
    ("2009 Intake", "https://docs.google.com/spreadsheets/d/11A67bHzjsJ2SSIyFIoK5MNbIdlSXtORDDtcHA_adcsg/edit?gid=831397415#gid=831397415"),
    ("2009 Outflow", "https://docs.google.com/spreadsheets/d/11A67bHzjsJ2SSIyFIoK5MNbIdlSXtORDDtcHA_adcsg/edit?gid=765780685#gid=765780685"),
    ("2008 Intake", "https://docs.google.com/spreadsheets/d/1Av7-Wr-Lf5G3lbjz6BOJfONgdPqfD_VyeG9eBWazU6A/edit?gid=938588532#gid=938588532"),
    ("2008 Outflow", "https://docs.google.com/spreadsheets/d/1Av7-Wr-Lf5G3lbjz6BOJfONgdPqfD_VyeG9eBWazU6A/edit?gid=1557235659#gid=1557235659"),
    ("2007", "https://docs.google.com/spreadsheets/d/1CvEIJbREKwJRMFqkTOFWd2u6Bb_JZzhwLkU1JHdiYFE/edit?gid=994387581#gid=994387581"),
    ("Licensing", "https://docs.google.com/spreadsheets/d/1q5PSJaDc-cpfKXBn6tQbg0jVxvwyrCNZUBk-5CCortA/edit?gid=531274251#gid=531274251"),
]

# Download and load each sheet as a DataFrame
dfs = {}
for year, url in sheet_links:
    csv_url = gsheet_to_csv_url(url)
    try:
        dfs[year] = pd.read_csv(csv_url)
        print(f"Loaded {year} data: {dfs[year].shape}")
    except Exception as e:
        print(f"Failed to load {year}: {e}")

# Example: display the first few rows of 2022 data
#dfs["2022"].head()

Loaded 2023 data: (347, 154)
Loaded 2022 data: (369, 154)
Loaded 2021 data: (354, 154)
Loaded 2020 data: (356, 154)
Loaded 2019 data: (349, 154)
Loaded 2018 data: (329, 173)
Loaded 2017 data: (279, 183)
Loaded 2016 data: (260, 204)
Loaded 2015 data: (257, 183)
Loaded 2014 data: (260, 213)
Loaded 2013 Intake data: (282, 47)
Loaded 2013 Outflow data: (282, 59)
Loaded 2012 Intake data: (282, 58)
Loaded 2012 Outflow data: (280, 57)
Loaded 2011 Intake data: (268, 58)
Loaded 2011 Outflow data: (269, 56)
Loaded 2010 Intake data: (264, 55)
Loaded 2010 Outflow data: (265, 57)
Loaded 2009 Intake data: (279, 66)
Loaded 2009 Outflow data: (280, 66)
Loaded 2008 Intake data: (276, 66)
Loaded 2008 Outflow data: (276, 67)
Loaded 2007 data: (297, 27)
Loaded Licensing data: (2974, 7)


Get Facility Location Info

In [ ]:
# --- Split 'location_1' column in 2016 dataset into city, latitude, longitude ---

def parse_location1(val):
    # Example: 'Denver, CO 80202\n(39.7392, -104.9903)'
    if pd.isnull(val):
        return pd.Series([None, None, None])
    # Split on newline
    parts = str(val).split('\n')
    city = parts[1]
    latlon = parts[2]
    # Extract lat, lon
    m2 = re.match(r'^\(([-\d.]+),\s*([-\d.]+)\)', latlon)
    lat = float(m2.group(1)) if m2 else None
    lon = float(m2.group(2)) if m2 else None
    return pd.Series([city, lat, lon])

if "2016" in dfs and "location_1" in dfs["2016"].columns:
    dfs["2016"][["city_from_loc", "lat", "lon"]] = dfs["2016"]["location_1"].apply(parse_location1)
    dfs["2016"]["city_from_loc"] = dfs["2016"]["city_from_loc"].str.rstrip(",")

# Show the result
dfs["2016"][["location_1", "city_from_loc", "lat", "lon"]].head(20)

,location_1,city_from_loc,lat,lon
0,"\nHartsel Colorado, \n(35.500801, -117.9478)","Hartsel Colorado,",35.500801,-117.947800
1,"\nLongmont, \n(40.165729, -105.101194)","Longmont,",40.165729,-105.101194
2,"\nLakewood, \n(39.710997, -105.088872)","Lakewood,",39.710997,-105.088872
3,NaN,None,NaN,NaN
4,NaN,None,NaN,NaN
5,NaN,None,NaN,NaN
6,"\nLittleton, \n(39.612653, -105.016198)","Littleton,",39.612653,-105.016198
7,"\nCedaredge, \n(38.900738, -107.923767)","Cedaredge,",38.900738,-107.923767
8,"\nDelta, \n(38.741684, -108.070175)","Delta,",38.741684,-108.070175
9,"\nFort Collins, \n(40.588972, -105.082459)","Fort Collins,",40.588972,-105.082459


In [ ]:
# --- Split 'location_1' column in 2017 dataset into city, zip code, latitude, longitude ---

def parse_location1_2017(val):
    # Example: 'Denver, CO 80202\n(39.7392, -104.9903)'
    if pd.isnull(val):
        return pd.Series([None, None, None, None])
    parts = str(val).split('\n')
    if len(parts) < 2:
        return pd.Series([None, None, None, None])
    # First part: 'Denver, CO 80202'
    city_zip = parts[0]
    # Second part: '(39.7392, -104.9903)'
    latlon = parts[1]
    # Extract city and zip
    city_zip_match = re.match(r'^(.*),\s*[A-Z]{2}\s*(\d{5})$', city_zip)
    if city_zip_match:
        city = city_zip_match.group(1)
        zip_code = city_zip_match.group(2)
    else:
        city = city_zip
        zip_code = None
    # Extract lat, lon
    latlon_match = re.match(r'^\(([-\d.]+),\s*([-\d.]+)\)', latlon)
    lat = float(latlon_match.group(1)) if latlon_match else None
    lon = float(latlon_match.group(2)) if latlon_match else None
    return pd.Series([city, zip_code, lat, lon])

if "2017" in dfs and "location_1" in dfs["2017"].columns:
    dfs["2017"][["city_from_loc", "zip_from_loc", "lat", "lon"]] = dfs["2017"]["location_1"].apply(parse_location1_2017)
    dfs["2017"]["city_from_loc"] = dfs["2017"]["city_from_loc"].str.rstrip(",")

# Show the result
dfs["2017"][["location_1", "city_from_loc", "zip_from_loc", "lat", "lon"]].head(20)

In [ ]:
dfs["2017"]["location_1"].head()

#dfs["2017"]["location_1"]

#2016 pacfa license zip code county
#2015 PACFA License Number Facility Physical Street Address Zip Code County
#2014 PACFA License Number Facility Street address Zip Code County

Index(['pacfa_license_number', 'footnotes', 'facility_name', 'zip_code',
       'county', 'adult_dogs', 'adult_dogs_beginning_count',
       'adult_dogs_beginning_foster_count', 'adult_dogs_stray',
       'adult_dogs_owner_relinquished',
       ...
       'other_owner_requested_euthanasia', 'other_ending_count',
       'other_foster_count', 'other_avg_los', 'other_notes', 'location_1',
       'city_from_loc', 'zip_from_loc', 'lat', 'lon'],
      dtype='object', length=208)

In [ ]:
# Mapping of possible column names for each field
license_no_cols = ['License Number', 'license_number', 'PACFA License Number', 'PACFA License No', 'License No']
facility_cols = ['Facility Name', 'facility_name', 'Facility', 'FACILITY NAME', 'Facility/Organization']
address_cols = ['Address', 'address', 'Facility Address', 'FACILITY ADDRESS', 'Street Address']
city_cols = ['City', 'city', 'Facility City', 'FACILITY CITY']
state_cols = ['State', 'state', 'Facility State', 'FACILITY STATE']
zip_cols = ['zip_code','Zip', 'zip', 'ZIP', 'Zip Code', 'ZIP Code', 'Facility Zip', 'FACILITY ZIP']
county_cols = ['County', 'county', 'Facility County', 'FACILITY COUNTY']

def find_column(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

records = []
for year, df in dfs.items():
    cols = df.columns
    lcol = find_column(cols, license_no_cols)
    fcol = find_column(cols, facility_cols)
    acol = find_column(cols, address_cols)
    ccol = find_column(cols, city_cols)
    scol = find_column(cols, state_cols)
    zcol = find_column(cols, zip_cols)
    kcol = find_column(cols, county_cols)
    if fcol is None:
        continue
    sub = pd.DataFrame()
    sub['year'] = [year] * len(df)
    sub['facility_name'] = df[fcol]
    sub['address'] = df[acol] if acol else None
    sub['city'] = df[ccol] if ccol else None
    sub['state'] = df[scol] if scol else None
    sub['county'] = df[kcol] if kcol else None
    records.append(sub)

facilities_df = pd.concat(records, ignore_index=True)
facilities_df = facilities_df.drop_duplicates().dropna(subset=['facility_name'])
facilities_df.reset_index(drop=True, inplace=True)

# Example: show first few rows
facilities_df.head()

,year,facility_name,address,city,state,county
0,2023,"2 Blondes All Breed Rescue, Inc.",None,None,None,None
1,2023,4 Paws 4 Life Rescue,None,None,None,None
2,2023,5280 Reptile Room North,None,None,None,None
3,2023,5280 Reptile Room west,None,None,None,None
4,2023,7 Paws Rescue Ranch,None,None,None,None
